<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 18 — מטריצת בלבול וסוגי טעויות
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
מסנן הודעות עלול לחסום הודעה חשובה או להחמיץ זבל. שתי הטעויות שונות במשמעותן, גם אם שתיהן נספרות כשגיאה.

## מטרות
- למלא מטריצה.
- להגדיר חיובי.
- לפרש שתי טעויות שונות.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## הנחיות למורה
זהו פתרון מתועד למעבדה. מבדילים בין תוצאה נכונה לבין נימוק נכון, ובודקים שהקוד תואם לנתוני התלמיד. המקורות הישנים נשמרו ללא שינוי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## פתרון חלק 1
הפתרון הבא תואם לנתוני התרגול העצמאי, ולא לנתוני הדוגמה הפתורה.

</div>

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt

In [ ]:
# דוגמה דידקטית מומצאת — מסווג קבוע שאינו מאומן
actual = ["OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK", "Defect", "Defect"]
predicted = ["OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK", "OK"]
pairs = pd.DataFrame({"actual": actual, "predicted": predicted})
display(pairs)

<div dir="rtl" lang="he" style="text-align: right;">

## 1–2. טבלה ואימות
שמונה תקינים סווגו כתקינים; שני פגומים סווגו כתקינים. אין חיזוי Defect. סדר התוויות המפורש שומר גם עמודה שכולה אפס.

</div>

In [ ]:
labels_binary = ["OK", "Defect"]
cm_binary = confusion_matrix(actual, predicted, labels=labels_binary)
table_binary = pd.DataFrame(
    cm_binary, index=labels_binary, columns=labels_binary
)
table_binary.index.name = "Actual"
table_binary.columns.name = "Predicted"
display(table_binary)

<div dir="rtl" lang="he" style="text-align: right;">

## 3. פירוש
החיובי Defect. TN=8, FP=0, FN=2, TP=0. האלכסון 8 וסך התאים 10, לכן Accuracy=0.8=80%. המסווג מחמיץ את כל הפגומים. הוא מסווג תמיד OK; הציון הכולל נשען על שכיחות התקינים. זו המחשה לימודית של בעיית חוסר איזון, לא תוצאת אימון על CSV.

</div>

In [ ]:
tn = table_binary.loc["OK", "OK"]
fp = table_binary.loc["OK", "Defect"]
fn = table_binary.loc["Defect", "OK"]
tp = table_binary.loc["Defect", "Defect"]
n = cm_binary.sum()
accuracy_from_matrix = (tn + tp) / n
print("TN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)
print("Total:", n, "Accuracy:", accuracy_from_matrix)
print("Accuracy from pairs:", accuracy_score(actual, predicted))

<div dir="rtl" lang="he" style="text-align: right;">

## 4. אותה הערכה על Penguins

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## ממשיכים את אותו מודל משיעור 24
התא הבא הוא **קוד הכנה נתון שכבר נלמד**, להרצה מלמעלה; אין צורך לכתוב אותו מחדש בשיעור זה.
הוא משחזר בדיוק את המודרכת: סנפיר ומסה, חלוקה 25%, seed=42, stratify, השלמה ותקנון מאימון בלבד ו־ K=3 קבוע. כעת מוסיפים דרך תיאור לתחזיות שכבר התקבלו; לא בוחרים פרמטרים לפי Test.
Penguins: עותק הקורס בן 344 שורות ו־ 7 עמודות. מקור: Kristen Gorman ו־ Palmer Station LTER; ייחוס ותיעוד [palmerpenguins](https://github.com/allisonhorst/palmerpenguins), Horst, Hill & Gorman (2020), DOI:10.5281/zenodo.3960218. התאמה לפרויקט גמר טעונה בדיקה נפרדת.

</div>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
display(df.head())
display(df.isna().sum())

model_df = df.dropna(subset=["species"]).copy()
features = ["flipper_length_mm","body_mass_g"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)

scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head())

model = KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="euclidean")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
print(predictions[:5])

comparison = pd.DataFrame(
    {"actual": y_test, "predicted": predictions}, index=y_test.index
)
comparison["correct"] = comparison["actual"] == comparison["predicted"]
display(comparison.head(10))
correct_count = comparison["correct"].sum()
accuracy = accuracy_score(y_test, predictions)
print("Correct:", correct_count, "out of", len(y_test))
print("Accuracy:", accuracy, "Percent:", 100 * accuracy)

In [ ]:
labels_species = ["Adelie", "Chinstrap", "Gentoo"]
cm_species = confusion_matrix(y_test, predictions, labels=labels_species)
table_species = pd.DataFrame(
    cm_species, index=labels_species, columns=labels_species
)
table_species.index.name = "Actual"
table_species.columns.name = "Predicted"
display(table_species)

# אלכסון ראשי: החיזוי זהה לאמת
correct_from_matrix = (
    table_species.loc["Adelie", "Adelie"]
    + table_species.loc["Chinstrap", "Chinstrap"]
    + table_species.loc["Gentoo", "Gentoo"]
)
print("N:", cm_species.sum())
print("Accuracy from diagonal:", correct_from_matrix / cm_species.sum())
print("Actual Chinstrap:", table_species.loc["Chinstrap"].sum())
print("Predicted Chinstrap:", table_species["Chinstrap"].sum())
print("Actual Chinstrap predicted Adelie:", table_species.loc["Chinstrap", "Adelie"])

chart = ConfusionMatrixDisplay(
    confusion_matrix=cm_species, display_labels=labels_species
)
chart.plot(cmap="Blues", values_format="d", colorbar=False)
plt.title("Penguins — fixed K=3; test split from lesson 24")
plt.tight_layout()
plt.show()

<div dir="rtl" lang="he" style="text-align: right;">

## 5. כרטיס הערכה
קראו את המספרים מפלט ההרצה: התא Chinstrap→Adelie הוא מספר דוגמאות Chinstrap שסווגו Adelie. התא Adelie→Chinstrap הוא סוג טעות הפוך; אין סיבה שספירותיהם יהיו זהות.
סכום שורת Chinstrap הוא מספר Chinstrap האמיתיים בבדיקה; סכום עמודתו הוא מספר כל התחזיות Chinstrap, כולל השגויות. סכום האלכסון מחולק ב־ 86 אמור לשחזר את ה־ Accuracy של שיעור 24.
מגבלה: חלוקה אחת ומדגם מוגבל אינם מוכיחים הכללה לאזור אחר. לפני סימון FN חובה לקבוע מחלקת מטרה; עבור Chinstrap מול היתר, החמצותיו הן כל התאים בשורתו שמחוץ לאלכסון. הבחירה הזאת אינה משנה את המטריצה הרב־מחלקתית המקורית.
דווחו בדוח הפרויקט על המודל והחלוקה, סדר התוויות, הצירים, ספירות ומגבלה. אין לבחור K מחדש לאור דוח זה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

### תוצאת בדיקה בעת הכנת הערכה
בעותק שנבדק: מטריצת Penguins בסדר Adelie, Chinstrap, Gentoo היא [[24,13,1], [12,5,0], [0,0,31]]. סך הכול 86 תצפיות ו־ 60 הצלחות (כ־ 69.77%). בשורת Chinstrap יש 17 דוגמאות: 5 נכונות ו־ 12 שסווגו Adelie; בעמודת Chinstrap יש 18 חיזויים: 5 נכונים ו־ 13 שהיו Adelie. אין לבלבל בין שתי הקבוצות. המספרים הם תוצאת ההרצה שנבדקה, ויש לקרוא מחדש את הפלט אם גרסת הנתונים או הספרייה משתנה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## תשובה לכרטיס היציאה
כי מחיר הטעויות שונה לפי השימוש. אחוז בלבד אינו מספר מי נפגע וכיצד.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## משוב דידקטי
בדקו בנפרד: בחירת פעולה, תקינות קוד, פירוש פלט, יחידות ומגבלה. שאלו מה ישתנה בקלט חדש. אם יש קושי, חזרו לדוגמה קטנה לפני הגדלת המאגר. זהו משוב מקומי, לא מחוון ניקוד רשמי.

</div>